# Coding Session 07: Solutions
## Did the visit work, or did the door just open?

These are the worked solutions to the four exercises, with the same Part 1 and setup as the student notebook so this file runs on its own from top to bottom. In Colab, save a copy to Drive before editing.

**Open in Colab:** [Student notebook](https://colab.research.google.com/github/albertostefanelli/DSPC_coding_sessions/blob/master/wk07_itt_vs_late/wk07_codingsession.ipynb) · [Solutions notebook](https://colab.research.google.com/github/albertostefanelli/DSPC_coding_sessions/blob/master/wk07_itt_vs_late/wk07_codingsession_solutions.ipynb).

## 1. What are we trying to measure?

### The story

In 1998, researchers at Yale took a list of registered voters in New Haven and randomly picked about one household in five to get a visit. Think of it as a weighted coin flip: heads, canvassers would try to visit that household before the election and ask the people there to vote; tails, nobody would visit. After the election, the researchers checked the public voter file to see who had voted. In this session, *turnout* means the share of people who voted.

The coin flip worked as planned. The visits did not go as planned. Canvassers tried every assigned address, and at about two thirds of them nobody answered the door. Those people were **assigned** a visit but never **received** one. We say a household was **reached** (or *contacted*, the word the data use) when someone in it opened the door to the canvasser. The two words mean the same thing here.

![Flow diagram. 13,383 voters are split by a coin flip into 2,650 assigned a visit and 10,733 control. The assigned group then splits into 869 people whose household answered the door and 1,781 who were never reached. The control group contains the same mix of would-answer and would-not-answer people, but we cannot tell who is who.](https://raw.githubusercontent.com/albertostefanelli/DSPC_coding_sessions/refs/heads/master/wk07_itt_vs_late/assets/fig1_who_ends_up_where.png)

*Figure 1. Where the 13,383 people in our sample ended up. The first split is a coin flip. The second split, inside the assigned group, is not.*

That second split is the whole problem. Whether a door opens is not random. People who are home in the evening, who open the door to a stranger, and who have lived at the same address for a while are also, on average, the kind of people who vote more. So the 869 people who were reached are not a random slice of the 2,650 who were assigned. They are the easy-to-reach ones.

Two words we will use a lot. The **estimand** is the exact thing we want to know, for example "the effect of a visit on the people who get one". The **estimate** is the number we compute from the data. For each number below, we will ask which estimand it actually answers.

### Three numbers you could compute

Here are three ways to turn the data into "the effect of canvassing". Each one compares some assigned people with some control people. Figure 2 shows which groups each comparison uses.

![Three panels. A: the naive comparison puts contacted people against all of control, +13.5 points. B: the ITT puts all assigned people against all of control, +3.95 points. C: the LATE puts contacted people against the would-have-been-reached part of control, +12.0 points.](https://raw.githubusercontent.com/albertostefanelli/DSPC_coding_sessions/refs/heads/master/wk07_itt_vs_late/assets/fig2_three_comparisons.png)

*Figure 2. The same four groups, compared three ways. Block heights show the share of people; the percentages inside are turnout.*

**A. The naive comparison: contacted people versus control.** Take the people whose household was actually reached and compare their turnout with the control group. In New Haven that gives about +13.5 percentage points (a percentage point is one point on the turnout scale: going from 44.2% to 57.7% is 13.5 points). It looks impressive, and it is biased. The reached group was picked *after* the coin flip, by whoever happened to open the door. The control group contains everybody, reachable or not. People who open the door differ from people who do not, so part of the gap was there before any canvasser showed up. That is selection, the Week 2 mistake, showing up inside an experiment.

**B. The intent-to-treat effect (ITT): everyone assigned versus everyone in control.** Keep every assigned person in the treatment group, whether or not their door opened, and compare with every control person. The coin flip created these two groups, so they are alike on average, and the difference is a fair estimate of what *assigning* a visit does. In New Haven the ITT is about +3.95 percentage points. This number is protected by the coin flip. Its weakness is that it mixes people who got a visit with people who did not, so it dilutes what a visit does to the people who actually get one.

**C. The local average treatment effect (LATE): the effect on the people a visit actually reaches.** If a visit only matters when it happens, then the whole ITT comes from the 32.8% of assigned people who were reached. Spread over everyone, the effect looks like +3.95. Concentrated on the third who got the visit, it is about three times larger, around +12 points. That larger number is the LATE. "Local" just means the effect holds only for the people whose door opens when a canvasser comes. The next two sections explain the division and what it assumes.

A note on names. **ATE** is short for *average treatment effect*. The ITT is an average treatment effect of **being assigned**. The LATE is an average treatment effect of **being reached**, but only for one group of people: those whose door opens when a canvasser comes. It says nothing about what a visit would do to people who never open the door.

### Compliance: did the treatment actually happen?

**Compliance** means that what a person received matches what they were assigned. **Noncompliance** is the gap between the two. It is rarely a refusal. In canvassing it is mostly bad luck: nobody home, a wrong address, a locked apartment building, or the campaign running out of days before the election.

The **compliance rate** answers one question: among the people assigned a visit, what share actually received one?

> Suppose 10 people are assigned a visit, and 4 of them live in a household where someone opened the door. The compliance rate is 4 / 10 = 0.40, or 40%. The denominator is all 10 assigned people, including the 6 who were not reached.

In Python, if `contacted` is a column of 0s and 1s (we will rename the data's `cntany` column to `contacted` when we load it), taking `.mean()` of the assigned rows gives exactly that fraction.

A campaign measures compliance by comparing its **assignment list** (who was supposed to get a visit) with its **contact records** (who was actually reached). Two rules keep this honest:

- **Decide in advance what counts as "received".** A knock on a closed door is not a contact. For a phone program, "the voter picked up" and "the voter heard the whole script" are different definitions, and they give different compliance rates. Pick one before the data come in.
- **A missing record is not a failed contact.** If a canvasser reports "knocked, nobody home", that person is coded 0. If there is no report at all, nobody knows what happened, and the value should stay missing. Quietly turning unknowns into 0s makes the program look worse at reaching people than it was.

In our New Haven data, 869 of the 2,650 assigned people lived in a household where someone was reached, so the compliance rate is 869 / 2,650 = 32.8%. One detail: contact is recorded for the whole household. If two people share a home and one of them opens the door, both are coded as contacted.

### Why divide the ITT by the compliance rate?

![A horizontal bar for the 2,650 assigned people. The red 32.8 percent were reached and get the full visit effect of 12 points. The gray 67.2 percent were never reached and get 0. Averaging gives 3.95 points, the ITT.](https://raw.githubusercontent.com/albertostefanelli/DSPC_coding_sessions/refs/heads/master/wk07_itt_vs_late/assets/fig3_why_divide.png)

*Figure 3. The ITT is the visit's effect averaged over everyone assigned, including the two thirds who got no visit.*

Think of the 2,650 assigned people as one group. The 869 who were reached got the visit's full effect. The other 1,781 got no visit, so a visit cannot have changed their turnout. The ITT averages over both:

ITT = 0.328 × (effect on the people reached) + 0.672 × 0

To get the effect on the people reached, undo the averaging:

**LATE = ITT ÷ compliance rate** = 3.95 ÷ 0.328 ≈ 12.0 percentage points

You can also count votes. An ITT of 3.95 points across 2,650 assigned people is about 105 extra votes. If all of those votes came from the 869 people who were reached, that is 105 / 869 ≈ 0.12, or 12 points per reached person. Both routes give 12 points.

### What has to be true for the LATE to make sense

The division is only fair if a few things hold. Some of them we can check in the data. Others, like the exclusion restriction below, we have to argue for.

- **Assignment was random.** The coin flip, not the canvassers, decided who was on the list. True by design here.
- **Assignment actually changed contact.** If assigned households were reached no more often than control households, there is nothing to divide by.
- **Assignment matters only through contact.** Being on the list changed nothing for people whose door never opened. A flyer left at the door would break this: it could change turnout without any recorded contact. This is called the **exclusion restriction**.
- **Nobody in control got a visit.** This is what lets the simple formula use the assigned group's contact rate as the denominator. If some control households had been visited anyway, we would divide by the *difference* in contact rates between the two groups. For example, 40% contact in treatment and 10% in control gives a denominator of 0.30, not 0.40. Zero contact in control also rules out the odd case of a household that would be reached only if it was *not* assigned.
- **Households did not affect each other.** One household's visit did not change its neighbor's turnout.

When these hold, the LATE describes **compliers**: people whose household gets reached if, and only if, they are assigned a visit. It does not describe people whose door never opens. They might react to a visit differently, and this experiment cannot tell us how.

**Units.** Turnout levels are percentages (44.2%). Differences between them are **percentage points (pp)**. Python gives you proportions (0.442); multiply by 100 to report either.

## Setup: the data: the 1998 New Haven experiment

The data come from Alan Gerber and Donald Green's field experiment (an experiment run in the real world, not in a lab) in New Haven, Connecticut, before the November 1998 election. About 30,000 registered voters were randomly assigned to door-to-door canvassing, mailings, phone calls, or nothing, and turnout was read from the official voter file afterwards. Of those, we keep the 13,383 who got no mailing or phone call and have a turnout record; the code below does this step by step. The study was published as Gerber and Green (2000), "The Effects of Canvassing, Telephone Calls, and Direct Mail on Voter Turnout: A Field Experiment", *American Political Science Review* 94(3): 653–663. The file we use is the one the authors released in 2005 so that others could redo the analysis; it corrected some errors in the original. The source and download link are in [data notes](https://github.com/albertostefanelli/DSPC_coding_sessions/blob/master/wk07_itt_vs_late/data/README.md) in this folder.

Each row is a **person**. People in the same household share one household ID, one assignment, and one contact record.

| Column | What it records | Values |
|---|---|---|
| `id1` | Household ID. Rows with the same `id1` live together. | number |
| `persngrp` | Was this household assigned a canvassing visit? | 1 = yes, 0 = control |
| `cntany` | Was anyone in this household reached by a canvasser? | 1 = yes, 0 = no |
| `vote98` | Did this person vote in November 1998? | 1 = yes, 0 = no, 99 = no record |
| `mailings` | How many mailings the household was assigned | 0, 1, 2, 3 |
| `phongotv` | Was the household assigned a get-out-the-vote phone call? | 1 = yes, 0 = no |
| `placebo` | Was the household assigned a placebo call (a call about something else)? | 1 = yes, 0 = no |

Two things to keep in mind. First, `cntany` is a household measure: if one person opened the door, everyone in that household is coded 1. Second, `vote98 == 99` means the voter file had no record for this person. It is not a zero, and we never treat it as one.

To keep the comparison clean, we use only households assigned **no mailings, no phone call, and no placebo call**. That leaves the visit as the only thing that differs between the assigned and control groups.

In [1]:
# Import the tools we need: pandas for tables, numpy for numbers, matplotlib for plots.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read the New Haven file straight from the course repository (you need to be online).
# Python joins the two quoted strings inside the parentheses into one address.
raw = pd.read_csv(
    'https://raw.githubusercontent.com/albertostefanelli/DSPC_coding_sessions/'
    'master/wk07_itt_vs_late/data/gerber_green_new_haven.csv'
)
print('Rows and columns:', raw.shape)
raw.head()

Rows and columns: (31098, 7)


,id1,persngrp,cntany,vote98,mailings,phongotv,placebo
0,13696,1,0,0,1,0,0
1,11930,1,0,0,2,0,0
2,14210,1,0,1,0,0,0
3,6818,1,0,0,0,0,0
4,7423,1,0,1,2,1,0


In [2]:
# Step 1: keep only households with no mailing, no phone call, and no placebo call.
# Each condition sits in parentheses; & means all three must be true.
df = raw[(raw['mailings'] == 0) & (raw['phongotv'] == 0) & (raw['placebo'] == 0)].copy()

# Step 2: vote98 == 99 means "no turnout record". That is unknown, not a non-vote,
# so we replace 99 with NaN (pandas' missing value) and drop those rows.
df['vote98'] = df['vote98'].replace(99, np.nan)
print('People with no turnout record:', df['vote98'].isna().sum())
df = df.dropna(subset=['vote98'])

# Step 3: rename the three columns we use so the code below reads in plain English.
df = df.rename(columns={
    'persngrp': 'assigned_treatment',
    'cntany': 'contacted',
    'vote98': 'turned_out',
})
print('People in the analysis:', len(df))
print('Households in the analysis:', df['id1'].nunique())
df.head()

People with no turnout record: 593
People in the analysis: 13383
Households in the analysis: 10067


,id1,assigned_treatment,contacted,turned_out,mailings,phongotv,placebo
2,14210,1,0,1.0,0,0,0
3,6818,1,0,0.0,0,0,0
9,5039,1,0,0.0,0,0,0
13,21203,1,0,0.0,0,0,0
14,20246,1,0,0.0,0,0,0


**About the 593 dropped rows.** Leaving out people with no turnout record is a normal choice, but it is a choice. It is safe as long as having a record has nothing to do with being assigned a visit. Here the share without a record is about 4% in both groups, which suggests that dropping them does not tip the comparison one way or the other. Note the difference from the compliance rule above: a missing turnout record is dropped, never counted as a non-vote.

In [3]:
# A mean of 0/1 turnout values is the turnout rate as a proportion.
# Group by randomized assignment, keeping assigned-but-not-reached people included.
turnout_by_assignment = df.groupby('assigned_treatment')['turned_out'].mean()
control_turnout = turnout_by_assignment.loc[0]
itt = turnout_by_assignment.loc[1] - control_turnout

# Compliance rate: people in reached households / all assigned people (869 / 2,650).
treatment_group = df[df['assigned_treatment'] == 1]
compliance_rate = treatment_group['contacted'].mean()
control_contact = df[df['assigned_treatment'] == 0]['contacted'].mean()

# The simple LATE formula needs zero contact in control.
assert control_contact == 0
late = itt / compliance_rate

# Observed turnout levels of the two assigned subgroups (not causal effects).
contact_turnout = treatment_group[treatment_group['contacted'] == 1]['turned_out'].mean()
noncontact_turnout = treatment_group[treatment_group['contacted'] == 0]['turned_out'].mean()

print(f'ITT: {100*itt:+.2f} pp')
print(f'Compliance rate: {compliance_rate:.1%}')
print(f'LATE: {100*late:+.2f} pp')

ITT: +3.95 pp
Compliance rate: 32.8%
LATE: +12.05 pp


### The same number, the long way: find the look-alikes in control

The LATE compares contacted people (57.7% turnout) with the control people who *would have been reached* if a canvasser had come. We cannot see who they are, but we can work out their turnout.

The control group is a mix: about 32.8% would-be-reached people and 67.2% would-not-be-reached people, the same mix as the assigned group, because the coin flip made the two groups alike. We know the control group's overall turnout, 44.2%. Now assume the would-not-be-reached control people vote like the assigned people who were not reached, 43.5%. That is the exclusion restriction from Part 1: no visit, no effect. With that assumption there is only one unknown left:

44.2% = 0.328 × (would-be-reached turnout) + 0.672 × 43.5%

Solve it, and the would-be-reached control people vote at about 45.6%. Contacted people minus their look-alikes: 57.7 − 45.6 ≈ 12.0 points, the same LATE as before.

The naive comparison instead subtracts *all* of control, 44.2%, which is pulled down by the hard-to-reach. So its 13.5 points includes a gap that would have been there without any visit.

In [4]:
import statsmodels.formula.api as smf

# The coefficient on assigned_treatment is the difference in means (the ITT).
m = smf.ols('turned_out ~ assigned_treatment', data=df).fit()

# Convert only the reported interval to percentage points.
# Keep itt and late as proportions for the vote and cost calculations.
lo, hi = 100 * m.conf_int().loc['assigned_treatment']
print(f'ITT 95% interval: [{lo:+.2f}, {hi:+.2f}] pp')

ITT 95% interval: [+1.84, +6.06] pp


### Exercise 1. Correct a causal claim

An analyst groups voters by household contact, then tells the field director: “This difference shows how much being contacted raised turnout.”

**Calculate:** run the analyst's code, then group by `assigned_treatment` instead. Store the corrected difference as `itt_fixed` and print both differences in percentage points.

**Interpret:** write a correction for the field director. Identify the people mixed together when `contacted == 0`, explain why the larger contact gap does not establish a causal effect, and state what the ITT estimates in this analysis.

In [5]:
# Reproduce the analyst's calculation to identify what it actually compares.
# contacted == 0 combines control people and assigned people not reached.
analyst_means = df.groupby('contacted')['turned_out'].mean()
analyst_gap = analyst_means.loc[1] - analyst_means.loc[0]

# Repair the comparison by grouping on the randomized assignment.
# All assigned people remain together, whether their household was reached or not.
fixed_means = df.groupby('assigned_treatment')['turned_out'].mean()
itt_fixed = fixed_means.loc[1] - fixed_means.loc[0]

# The calculations above use proportions; multiply by 100 only for reporting.
print(f'Contact-based gap: {100*analyst_gap:+.2f} pp')
print(f'Assignment-based ITT: {100*itt_fixed:+.2f} pp')

Contact-based gap: +13.59 pp
Assignment-based ITT: +3.95 pp


**Interpretation:** The contact-based gap is **+13.59 pp**, but its comparison group mixes controls with assigned people in households that were not reached. Reachable households may have voted more often anyway, so this gap does not establish the causal effect of contact. The assignment comparison estimates a **+3.95 pp effect of assigning a visit**.

### Exercise 2. High turnout or a large treatment effect?

An analyst gives the contacted and noncontacted assigned groups equal weight when averaging turnout. A field director also argues: “The contacted group votes more often, so targeting similar households must produce the biggest turnout gain.”

**Calculate:** use `compliance_rate` and `1 - compliance_rate` to obtain the correctly weighted assigned-group turnout, `weighted_turnout`. Subtract `control_turnout` to get `itt_weighted`, and print it beside `itt`.

**Interpret:** explain which group the equal-weight average overrepresents. Then assess the targeting claim: does a high observed turnout rate show that contact changed that group's behavior the most? Distinguish a turnout level from a causal effect.

In [6]:
# An unweighted average of two group means gives each group a 50% share.
# That would represent the full assigned group only if the groups were equal in size.
equal_weight_turnout = (contact_turnout + noncontact_turnout) / 2

# Weight by shares of people, not shares of households.
# These weights add to 1 and include everyone assigned with complete records.
weighted_turnout = (
    compliance_rate * contact_turnout
    + (1 - compliance_rate) * noncontact_turnout
)

# Subtract the full control group's turnout to recover the assignment comparison.
itt_weighted = weighted_turnout - control_turnout

# Levels are percentages; differences are percentage points.
print(f'Actual contacted-group share: {compliance_rate:.1%}')
print(f'Equal-weight turnout: {100*equal_weight_turnout:.2f}%')
print(f'Correctly weighted turnout: {100*weighted_turnout:.2f}%')
print(f'Weighted ITT: {100*itt_weighted:+.2f} pp')
print(f'ITT from assignment means: {100*itt:+.2f} pp')

Actual contacted-group share: 32.8%
Equal-weight turnout: 50.56%
Correctly weighted turnout: 48.11%
Weighted ITT: +3.95 pp
ITT from assignment means: +3.95 pp


**Interpretation:** Equal weighting gives people in contacted households 50% of the weight even though they are only **32.8%** of the assigned group. Correct weighting gives **48.11% turnout** and a **+3.95 pp ITT**. High turnout among contacted households does not show that they gained most from contact: they might have voted anyway, and these observed rates do not compare causal effects across groups.

### Exercise 3. Is the cheaper offer a better decision?

Two hypothetical offers deliver the **same program to 1,000 similar assigned people**. Offer A charges **4 dollars per assigned person**. Offer B charges **12 dollars per person living in a contacted household**, whether or not that person was personally spoken to. B's fee uses the household-contact measure in these data.

For the calculation, assume the observed contact share and estimated effects carry over unchanged under either offer.

**Calculate:** use `n_next * itt` for expected additional voters. Compute each total bill and cost per additional voter; B bills for `n_next * compliance_rate` people.

**Interpret:** recommend an offer under these assumptions. Explain why dividing A's price per assigned person by `late` would use the wrong population. Give one reason this cost comparison does not guarantee the same result in a new campaign.

In [7]:
# Hypothetical prices for identical delivery and the same target population.
n_next = 1_000
price_assigned = 4
price_in_contacted_household = 12

# The ITT applies to every assigned person, including those not reached.
# Multiplying a proportion by the number assigned gives expected additional voters.
# Keep the fractional expectation; rounding early would distort cost per voter.
extra_voters = n_next * itt

# Offer A bills everyone assigned. Offer B bills people in contacted households.
# B does not bill per household or per verified conversation with an individual.
expected_billed_people = n_next * compliance_rate
bill_a = n_next * price_assigned
bill_b = expected_billed_people * price_in_contacted_household

# Both offers are assumed to deliver the same program, so the vote denominator is equal.
# Costs per additional voter are meaningful here because the estimated ITT is positive.
cpv_a = bill_a / extra_voters
cpv_b = bill_b / extra_voters

print(f'Expected additional voters: {extra_voters:.2f}')
print(f'Expected people billed under B: {expected_billed_people:.2f}')
print(f'Offer A bill: {bill_a:,.2f} dollars')
print(f'Offer B bill: {bill_b:,.2f} dollars')
print(f'Offer A: {cpv_a:.2f} dollars per additional voter')
print(f'Offer B: {cpv_b:.2f} dollars per additional voter')

Expected additional voters: 39.50
Expected people billed under B: 327.92
Offer A bill: 4,000.00 dollars
Offer B bill: 3,935.09 dollars
Offer A: 101.26 dollars per additional voter
Offer B: 99.61 dollars per additional voter


**Interpretation:** Under the stated assumptions, choose **B**, at about **99.61 dollars per additional voter**, versus **101.26 dollars for A**. A bills everyone assigned, so its price per person must be divided by the ITT; dividing by LATE would incorrectly apply the complier effect to everyone assigned. This is a conditional cost comparison, not a guarantee: the contact share, effect, or people reached could differ in a new campaign.

### Exercise 4. What does the interval establish?

Two reports claim LATEs of **30 percentage points** and **10 percentage points** for the same New Haven sample and household-contact measure. A manager says: "If a claim fits the confidence interval, we have proved that effect for every voter."

**Calculate:** multiply each claimed LATE by `compliance_rate` to express it as an implied ITT. Print both beside the ITT interval (`lo`, `hi`) from Part 2d. Treat the observed contact share as fixed for this rough comparison; this is not a full confidence interval for the LATE.

**Interpret:** explain how the interval bears on each claim, then correct the manager's statement. Address both uncertainty about the exact effect and which people a LATE describes.

In [8]:
# Both claims are in percentage points, not proportions.
large_late_claim_pp = 30
small_late_claim_pp = 10

# Under exclusion and the other stated assumptions, ITT = contact share * LATE.
# Treating the observed contact share as fixed gives this rough interval comparison.
# Multiplying percentage points by a share leaves the answer in percentage points.
implied_large_pp = compliance_rate * large_late_claim_pp
implied_small_pp = compliance_rate * small_late_claim_pp

# lo and hi are already in percentage points; no further conversion is needed.
print(f'ITT interval: [{lo:+.2f}, {hi:+.2f}] pp')
print(f'30 pp LATE implies ITT: {implied_large_pp:+.2f} pp')
print(f'10 pp LATE implies ITT: {implied_small_pp:+.2f} pp')

# Inclusion describes compatibility in this approximate comparison, not proof.
# This is not a full LATE interval: uncertainty in the contact share is held fixed.
large_inside = lo <= implied_large_pp <= hi
small_inside = lo <= implied_small_pp <= hi
print(f'30 pp claim fits this interval comparison: {large_inside}')
print(f'10 pp claim fits this interval comparison: {small_inside}')

ITT interval: [+1.84, +6.06] pp
30 pp LATE implies ITT: +9.84 pp
10 pp LATE implies ITT: +3.28 pp
30 pp claim fits this interval comparison: False
10 pp claim fits this interval comparison: True


**Interpretation:** The 30-point claim implies a **+9.84 pp ITT**, outside the interval **[+1.84, +6.06] pp**; the 10-point claim implies **+3.28 pp**, inside it. This rough check weighs against the larger claim and is compatible with the smaller one, but it does not prove an exact 10-point effect. A LATE describes people whose household gets reached because of assignment, so it need not apply to every voter, or to a new campaign's compliers.